# Part 4 — Entity profiles and field conflicts

Use the accepted Part 3 links to collect the Part 2 gold claims for each `entity_id`.

- Each claim has **C = R × L × F**. R is source reliability, L is entity link confidence, and F is the Part 2 confidence that the field was mapped correctly.
- One normalized value: use that value with **C = min(1, sum of claim scores)**. For a single claim this is simply R × L × F.
- Different normalized values: leave the profile field and its confidence **null**. Preserve every claim and report each alternative's **sum(RLF) / sum(RLF for all alternatives)**.
- `ai_decision`, `data_analyser_decision`, and `final_verdict` stay **null** for future review. No decision prompts or AI calls in this part.

R, L and F remain separate in the claim output. C and conflict shares are the requested scoring rules, not measured probabilities. Dates, licences and derivation are preserved; they do not add another scoring factor.

**Run all cells** to rebuild the outputs from the current database. Accepted links are required; possible matches and unlinked observations remain in Part 3.

Database handling retries idle notebook locks automatically. An actively used database is left alone; exported inputs are used when available, otherwise the run pauses with a status message. Connections close after reads and saves, including failed operations.

## Imports and settings

In [1]:
import hashlib
import json
import math
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import duckdb
from firmable.database import connect_database, close_database
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "firmable.duckdb"
PART4_DIR = PROJECT_ROOT / "outputs" / "part4"
PART4_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_ROWS = 5

with (PROJECT_ROOT / "resources" / "firmable_ontology.yaml").open() as file:
    ontology = yaml.safe_load(file)

# Use the same fields as Part 2; no dataset-specific extraction code.
FIELD_MAP = {
    f"{section}.{field}": f"{section}_{field}"
    for section in ("entity", "address")
    for field in ontology[section]
}
PROFILE_FIELDS = {
    field: column.removeprefix("entity_") for field, column in FIELD_MAP.items()
}
DECISION_COLUMNS = ["ai_decision", "data_analyser_decision", "final_verdict"]


def stage(name):
    print(f"\nPART 4 — {name}")

GOLD_COPY_DIR = PROJECT_ROOT / "outputs" / "part2" / "gold_inspection"
PART3_DIR = PROJECT_ROOT / "outputs" / "part3"
PART4_READY = False
PART4_INPUT_MODE = "current_database"


## Normalize values and calculate claim scores

Names and addresses use uppercase, collapsed whitespace, and cosmetic comma/full-stop spacing. Other punctuation, address unit separators, leading identifier zeroes and meaningful words are preserved. This does not do fuzzy matching or shorten legal names.

Website schemes and a leading `www.` are ignored for comparison; paths and queries remain significant. Dates use ISO dates; enum case is standardized. Nulls are not claims. A stated `unknown` remains a claim.

Missing, non-finite or out-of-range confidence stops the run with the source, record and field named. It never silently becomes 1. If all conflict support scores are zero, relative scores remain null.

In [2]:
def text_value(value):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    return None if value.casefold() in {"", "none", "null", "nan", "<na>", "n/a"} else value


def json_object(value, context):
    if value is None or (not isinstance(value, dict) and pd.isna(value)):
        return {}
    try:
        result = value if isinstance(value, dict) else json.loads(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"Invalid JSON for {context}") from error
    if not isinstance(result, dict):
        raise ValueError(f"Expected a JSON object for {context}")
    return result


def stable_id(*parts):
    return hashlib.sha256(json.dumps(parts, ensure_ascii=False).encode()).hexdigest()


def normalize_value(field, value):
    value = text_value(value)
    if value is None:
        return None
    value = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value)).strip()
    if field in {"entity.abn", "entity.acn", "entity.nzbn", "address.postcode"}:
        return re.sub(r"[\s-]", "", value)
    if field == "entity.date_registered":
        try:
            return pd.Timestamp(value).date().isoformat()
        except ValueError:
            return value  # Keep the claim; do not guess an invalid date.
    if field == "entity.website":
        parsed = urlsplit(value if "://" in value else "https://" + value)
        host = (parsed.hostname or "").casefold().removeprefix("www.")
        if not host:
            return value
        try:
            port = parsed.port
        except ValueError:
            return value
        host += f":{port}" if port not in {None, 80, 443} else ""
        path = "" if parsed.path in {"", "/"} else parsed.path
        return host + path + ("?" + parsed.query if parsed.query else "") + (
            "#" + parsed.fragment if parsed.fragment else "")
    if field in {"entity.entity_type", "entity.status"}:
        return value.casefold()
    if field in {"entity.legal_name", "entity.trading_name", "address.full", "address.locality"}:
        value = re.sub(r"[.,]", " ", value)
        value = re.sub(r"\s+", " ", value).strip()
    return value.upper()


def confidence_value(value, component, context):
    try:
        number = float(value)
    except (TypeError, ValueError):
        number = float("nan")
    if isinstance(value, bool) or not math.isfinite(number) or not 0 <= number <= 1:
        raise ValueError(f"Missing or invalid {component} for {context}: {value!r}. Check Part 2/3 output.")
    return number


def claim_score(reliability, link_confidence, field_confidence, context):
    return (
        confidence_value(reliability, "R", context)
        * confidence_value(link_confidence, "L", context)
        * confidence_value(field_confidence, "F", context)
    )

## Load current gold tables and accepted links

Join by `source_id` and `source_record_id`, then check that each accepted link has exactly one source observation and an existing entity. Identical duplicate observations count once; differing duplicates require Part 3 to be rerun.

The identity registry stays unchanged. An ACN-derived `entity_id` is not itself a stated ABN claim.

In [3]:
def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


def load_inputs():
    global PART4_INPUT_MODE
    connection = connect_database(DB_PATH) if DB_PATH.exists() else None
    if connection is None:
        copies = sorted(GOLD_COPY_DIR.glob("gold_*.parquet"))
        pool_path = PART3_DIR / "entities_pool.parquet"
        link_path = PART3_DIR / "entity_link_index.parquet"
        if not copies or not pool_path.exists() or not link_path.exists():
            print("Part 4 paused: database unavailable and complete exported inputs are missing. Existing results are preserved.")
            return None, None, None
        try:
            frames = []
            for path in copies:
                frame = pd.read_parquet(path)
                frame["gold_table"] = path.stem
                frames.append(frame)
            PART4_INPUT_MODE = "exported_files"
            print("Reading exported gold files and Part 3 links. Current database access will be retried when saving.")
            return pd.concat(frames, ignore_index=True), pd.read_parquet(pool_path), pd.read_parquet(link_path)
        except Exception as error:
            print("Part 4 paused: exported inputs could not be read:", error)
            return None, None, None
    try:
        tables = connection.execute("""
            SELECT table_schema, table_name FROM information_schema.tables
            WHERE table_type = 'BASE TABLE' AND starts_with(table_name, 'gold_')
            ORDER BY table_schema, table_name
        """).fetchall()
        if not tables:
            print("Part 4 paused: no approved gold tables. Existing results are preserved.")
            return None, None, None
        frames = []
        for schema, table in tables:
            frame = connection.execute(f"SELECT * FROM {quote_identifier(schema)}.{quote_identifier(table)}").df()
            frame["gold_table"] = table
            frames.append(frame)
        pool = connection.execute("SELECT * FROM entities_pool ORDER BY entity_id").df()
        links = connection.execute("SELECT * FROM entity_link_index ORDER BY link_id").df()
        PART4_INPUT_MODE = "current_database"
        return pd.concat(frames, ignore_index=True), pool, links
    except duckdb.Error as error:
        print("Part 4 paused: Part 3 inputs are not ready:", error)
        return None, None, None
    finally:
        close_database(connection)


def attach_links(gold, pool, links):
    keys = ["source_id", "source_record_id"]
    if pool["entity_id"].isna().any() or pool["entity_id"].duplicated().any():
        raise ValueError("Part 3 entities_pool must have unique, non-null entity IDs.")
    if links[keys + ["entity_id", "link_id"]].isna().any().any():
        raise ValueError("Part 3 links contain missing keys.")
    if links.duplicated(keys).any() or links["link_id"].duplicated().any():
        raise ValueError("Part 3 contains duplicate accepted links; rerun Part 3.")
    if not links["entity_id"].isin(pool["entity_id"]).all():
        raise ValueError("An accepted link points to an entity outside entities_pool.")
    observations = gold.rename(columns={
        "observation_source_id": "source_id",
        "observation_source_record_id": "source_record_id",
    }).copy()
    observations[keys] = observations[keys].astype("string")
    # Unlinked records cannot influence profiles or conflict scores.
    observations = observations.merge(links[keys], on=keys, how="inner", validate="many_to_one")
    payload = [column for column in observations if column != "gold_table"]
    observations = observations.sort_values("gold_table").drop_duplicates(payload)
    if observations.duplicated(keys).any():
        raise ValueError("Differing gold observations share an accepted source key; rerun Part 3.")
    linked = links.merge(observations, on=keys, how="left", validate="one_to_one", indicator=True)
    if (linked["_merge"] != "both").any():
        missing = linked.loc[linked["_merge"] != "both", keys].head(3).to_dict("records")
        raise ValueError(f"Accepted links have no current gold observation: {missing}. Rerun Part 3.")
    return linked.drop(columns="_merge")

## Preserve every linked field claim

In [4]:
CLAIM_COLUMNS = [
    "claim_id", "entity_id", "field_name", "raw_value", "normalized_value",
    "source_id", "source_record_id", "gold_table", "link_id", "match_method", "match_evidence",
    "source_reliability", "link_confidence", "field_confidence", "claim_score", "derivation_level",
    "observed_at", "ingested_at", "valid_from", "valid_to", "licence", "extractor_version", "conflict_id",
]


def build_claims(linked):
    claims = []
    for item in linked.to_dict("records"):
        context = f"{item['gold_table']} / record {item['source_record_id']}"
        field_scores = json_object(item.get("field_confidence"), context + " field confidence")
        derivations = json_object(item.get("derivation_level"), context + " derivation")
        for field, column in FIELD_MAP.items():
            raw = text_value(item.get(column))
            if raw is None:
                continue
            reliability = item.get("confidence_source_reliability")
            link_confidence = item["link_confidence"]
            field_confidence = field_scores.get(field)
            score = claim_score(reliability, link_confidence, field_confidence, context + " / " + field)
            claims.append({
                "claim_id": stable_id(item["link_id"], field),
                "entity_id": item["entity_id"], "field_name": field,
                "raw_value": raw, "normalized_value": normalize_value(field, raw),
                "source_id": item["source_id"], "source_record_id": item["source_record_id"],
                "gold_table": item["gold_table"], "link_id": item["link_id"],
                "match_method": item["match_method"], "match_evidence": item["match_evidence"],
                "source_reliability": float(reliability), "link_confidence": float(link_confidence),
                "field_confidence": float(field_confidence), "claim_score": score,
                "derivation_level": derivations.get(field),
                **{name: text_value(item.get("observation_" + name)) for name in [
                    "observed_at", "ingested_at", "valid_from", "valid_to", "licence", "extractor_version"
                ]},
                "conflict_id": None,
            })
    return pd.DataFrame(claims, columns=CLAIM_COLUMNS)

## Agreeing fields and conflicts

`firmable_entities` has one row per existing entity, with a value and a C column for each field. `entity_field_scores` explains uncontested field totals. `entity_field_claims` keeps the raw claims and provenance for both agreements and conflicts.

`entity_field_conflicts` has one row per conflicting entity/field with the three null decision columns. `entity_conflict_values` contains the alternatives, summed support and relative scores. Join these by `conflict_id`; join to the original claims by the same ID.

For example, support 2 for value A and support 1 for value B gives 2/3 and 1/3. Neither value is selected. Repeated notebook runs rebuild these outputs, so future decision editing needs its own save/review workflow before being enabled.

In [5]:
SCORE_COLUMNS = [
    "entity_id", "field_name", "value", "confidence", "support_score",
    "claim_count", "source_count", "status",
]
CONFLICT_COLUMNS = [
    "conflict_id", "entity_id", "field_name", "alternative_count", "total_support",
    "claim_count", *DECISION_COLUMNS,
]
VALUE_COLUMNS = [
    "conflict_id", "entity_id", "field_name", "alternative_id", "value",
    "support_score", "relative_score", "claim_count", "source_count",
]


def reconcile_fields(pool, claims):
    if claims["claim_id"].duplicated().any():
        raise ValueError("Repeated field claim IDs would inflate support scores.")
    if not claims["entity_id"].isin(pool["entity_id"]).all():
        raise ValueError("A field claim points outside entities_pool.")
    profiles = {
        entity_id: {
            "entity_id": entity_id,
            **{column: None for column in PROFILE_FIELDS.values()},
            **{column + "_confidence": None for column in PROFILE_FIELDS.values()},
            "has_conflict": False, "conflict_count": 0, "resolved_field_count": 0,
        }
        for entity_id in pool["entity_id"].sort_values()
    }
    claims = claims.copy()
    scores, conflicts, alternatives = [], [], []
    for (entity_id, field), group in claims.groupby(["entity_id", "field_name"], sort=True):
        values = group.groupby("normalized_value", sort=True)["claim_score"].sum()
        if len(values) == 1:
            value, support = next(iter(values.items()))
            confidence = min(1.0, float(support))
            column = PROFILE_FIELDS[field]
            profiles[entity_id][column] = value
            profiles[entity_id][column + "_confidence"] = confidence
            profiles[entity_id]["resolved_field_count"] += 1
            scores.append({
                "entity_id": entity_id, "field_name": field, "value": value,
                "confidence": confidence, "support_score": float(support),
                "claim_count": len(group), "source_count": group["source_id"].nunique(),
                "status": "single_claim" if len(group) == 1 else "agreement",
            })
            continue
        conflict_id = stable_id(entity_id, field)
        total = float(values.sum())  # Do not cap competing support before calculating shares.
        claims.loc[group.index, "conflict_id"] = conflict_id
        profiles[entity_id]["has_conflict"] = True
        profiles[entity_id]["conflict_count"] += 1
        conflicts.append({
            "conflict_id": conflict_id, "entity_id": entity_id, "field_name": field,
            "alternative_count": len(values), "total_support": total, "claim_count": len(group),
            **{column: None for column in DECISION_COLUMNS},
        })
        for number, (value, support) in enumerate(values.items(), 1):
            supporting = group[group["normalized_value"] == value]
            alternatives.append({
                "conflict_id": conflict_id, "entity_id": entity_id, "field_name": field,
                "alternative_id": f"k{number}", "value": value,
                "support_score": float(support),
                "relative_score": float(support) / total if total > 0 else None,
                "claim_count": len(supporting), "source_count": supporting["source_id"].nunique(),
            })
    profile_columns = [
        "entity_id", *PROFILE_FIELDS.values(),
        *(column + "_confidence" for column in PROFILE_FIELDS.values()),
        "has_conflict", "conflict_count", "resolved_field_count", "missing_fields",
    ]
    for profile in profiles.values():
        profile["missing_fields"] = json.dumps([
            field for field, column in PROFILE_FIELDS.items() if profile[column] is None
        ])
    return {
        "firmable_entities": pd.DataFrame(profiles.values(), columns=profile_columns),
        "entity_field_scores": pd.DataFrame(scores, columns=SCORE_COLUMNS),
        "entity_field_claims": claims,
        "entity_field_conflicts": pd.DataFrame(conflicts, columns=CONFLICT_COLUMNS),
        "entity_conflict_values": pd.DataFrame(alternatives, columns=VALUE_COLUMNS),
    }

## Save the five outputs to DuckDB, Parquet and CSV

In [6]:
def output_types(frame):
    # Give all-null and empty outputs stable SQL/Parquet types too.
    frame = frame.copy()
    integer_columns = {"claim_count", "source_count", "alternative_count", "conflict_count", "resolved_field_count"}
    float_columns = {
        "source_reliability", "link_confidence", "field_confidence", "claim_score",
        "confidence", "support_score", "total_support", "relative_score",
    }
    for column in frame:
        if column in integer_columns:
            frame[column] = frame[column].astype("Int64")
        elif column in float_columns or column.endswith("_confidence"):
            frame[column] = frame[column].astype("Float64")
        elif column == "has_conflict":
            frame[column] = frame[column].astype("boolean")
        else:
            frame[column] = frame[column].astype("string")
    return frame


def save_outputs(outputs, metrics):
    database_written = False
    connection = connect_database(DB_PATH)
    if connection is not None:
        try:
            connection.execute("BEGIN TRANSACTION")
            for table, frame in outputs.items():
                connection.register("_part4_output", frame)
                connection.execute(f"CREATE OR REPLACE TABLE {quote_identifier(table)} AS SELECT * FROM _part4_output")
                connection.unregister("_part4_output")
            connection.execute("COMMIT")
            database_written = True
        except duckdb.Error as error:
            try:
                connection.execute("ROLLBACK")
            except duckdb.Error:
                pass
            print("Part 4 database save stopped safely; saving output files:", error)
        finally:
            close_database(connection)
    for table, frame in outputs.items():
        frame.to_parquet(PART4_DIR / f"{table}.parquet", index=False)
        frame.to_csv(PART4_DIR / f"{table}.csv", index=False)
    metrics["database_written"] = database_written
    (PART4_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    if not database_written:
        print("Part 4 files saved. DuckDB persistence is pending; rerun when database access is available.")
    return database_written


## Run Part 4 and show the results

In [7]:
stage("Load gold observations and accepted entity links")
gold_observations, entities_pool, entity_link_index = load_inputs()
PART4_READY = gold_observations is not None
if PART4_READY:
    try:
        linked_observations = attach_links(gold_observations, entities_pool, entity_link_index)
        print(f"Gold observations: {len(gold_observations):,}")
        print(f"Accepted linked observations: {len(linked_observations):,}")
        print(f"Existing entities: {len(entities_pool):,}")
        print(f"Unlinked gold observations stay in Part 3: {len(gold_observations) - len(linked_observations):,}")

        stage("Calculate R × L × F for each field claim")
        field_claims = build_claims(linked_observations)
        print(f"Preserved field claims: {len(field_claims):,}")

        stage("Combine agreements and preserve conflicts")
        part4_outputs = {
            table: output_types(frame)
            for table, frame in reconcile_fields(entities_pool, field_claims).items()
        }
        firmable_entities = part4_outputs["firmable_entities"]
        entity_field_scores = part4_outputs["entity_field_scores"]
        entity_field_claims = part4_outputs["entity_field_claims"]
        entity_field_conflicts = part4_outputs["entity_field_conflicts"]
        entity_conflict_values = part4_outputs["entity_conflict_values"]
        run_metrics = {
            "run_at": datetime.now(timezone.utc).isoformat(),
            "input_mode": PART4_INPUT_MODE,
            "gold_observations": len(gold_observations),
            "linked_observations": len(linked_observations),
            **{table: len(frame) for table, frame in part4_outputs.items()},
            "profiles_with_conflicts": int(firmable_entities["has_conflict"].sum()),
            "single_claim_fields": int((entity_field_scores["status"] == "single_claim").sum()),
            "agreeing_fields": int((entity_field_scores["status"] == "agreement").sum()),
            "zero_support_conflicts": int((entity_field_conflicts["total_support"] == 0).sum()),
        }
        print(f"Entity profiles: {len(firmable_entities):,}")
        print(f"Uncontested fields: {len(entity_field_scores):,}")
        print(f"  Single claim: {run_metrics['single_claim_fields']:,}")
        print(f"  Multiple agreeing claims: {run_metrics['agreeing_fields']:,}")
        print(f"Conflicting fields: {len(entity_field_conflicts):,}")
        print(f"Profiles with conflicts: {run_metrics['profiles_with_conflicts']:,}")
        print("Conflicted profile values and confidences are null; all three decisions are null.")

        stage("Save Part 4 tables and files")
        database_written = save_outputs(part4_outputs, run_metrics)
        for table, frame in part4_outputs.items():
            print(f"{table}: {len(frame):,} rows — saved as Parquet and CSV")
        print("DuckDB tables saved:", database_written)
        print(f"Output folder: {PART4_DIR}")

        stage("Entity profile preview — values and calculated field confidence C")
        display(firmable_entities[[
            "entity_id", "legal_name", "legal_name_confidence", "abn", "acn", "has_conflict"
        ]].head(PREVIEW_ROWS))

        stage("Agreement example — supporting claims retain R, L and F")
        agreement = entity_field_scores[entity_field_scores["status"] == "agreement"].head(1)
        if agreement.empty:
            print("No multiple-claim agreements in the current inputs.")
        else:
            example = agreement.iloc[0]
            print(f"Entity {example['entity_id']} / {example['field_name']}: {example['value']}")
            print(f"Support sum = {example['support_score']:.4f}; final C = {example['confidence']:.4f}")
            evidence = entity_field_claims[
                (entity_field_claims["entity_id"] == example["entity_id"])
                & (entity_field_claims["field_name"] == example["field_name"])
            ]
            display(evidence[["gold_table", "raw_value", "source_reliability", "link_confidence", "field_confidence", "claim_score"]])
    except Exception as error:
        PART4_READY = False
        print(f"PART 4 — Run stopped safely: {type(error).__name__}: {error}")
        print("Resolve the reported input issue and rerun; incomplete profiles were not selected.")



PART 4 — Load gold observations and accepted entity links
Gold observations: 6,000
Accepted linked observations: 3,959
Existing entities: 3,415
Unlinked gold observations stay in Part 3: 2,041

PART 4 — Calculate R × L × F for each field claim
Preserved field claims: 20,865

PART 4 — Combine agreements and preserve conflicts
Entity profiles: 3,415
Uncontested fields: 17,624
  Single claim: 16,187
  Multiple agreeing claims: 1,437
Conflicting fields: 0
Profiles with conflicts: 0
Conflicted profile values and confidences are null; all three decisions are null.

PART 4 — Save Part 4 tables and files
firmable_entities: 3,415 rows — saved as Parquet and CSV
entity_field_scores: 17,624 rows — saved as Parquet and CSV
entity_field_claims: 20,865 rows — saved as Parquet and CSV
entity_field_conflicts: 0 rows — saved as Parquet and CSV
entity_conflict_values: 0 rows — saved as Parquet and CSV
DuckDB tables saved: True
Output folder: /Users/ario/Documents/firmable/outputs/part4

PART 4 — Entity

,entity_id,legal_name,legal_name_confidence,abn,acn,has_conflict
0,11000000948,QBE INSURANCE (INTERNATIONAL) PTY LIMITED,1.0,11000000948,000000948,False
1,11000003314,NEWCASTLE GOLF CLUB LTD,0.882,11000003314,000003314,False
2,11000003378,BBC HARDWARE LIMITED,1.0,11000003378,000003378,False
3,11000007876,AMPOL PETROLEUM PTY LTD,1.0,11000007876,000007876,False
4,11000013098,SYDNEY NIGHT PATROL & INQUIRY CO PTY LTD,0.882,11000013098,000013098,False



PART 4 — Agreement example — supporting claims retain R, L and F
Entity 11000000948 / address.country: AU
Support sum = 3.9200; final C = 1.0000


,gold_table,raw_value,source_reliability,link_confidence,field_confidence,claim_score
2634,gold_asic_company,AU,0.98,1.0,1.0,0.98
5786,gold_asic_company,AU,0.98,1.0,1.0,0.98
17555,gold_asic_company,AU,0.98,1.0,1.0,0.98
19671,gold_asic_company,AU,0.98,1.0,1.0,0.98


## Worked conflict example — illustration only

The real extracted sample currently has no field conflicts. This small in-memory example demonstrates the requested rule and is **not saved into the actual output tables**.

Two claims for `10 King St` score 0.8 each, and one for `12 King St` scores 0.4. Their support scores are 1.6 and 0.4; relative scores are **0.8 and 0.2**. Both the address value and its confidence remain null in the profile; all three decisions stay null.

In [8]:
stage("Worked conflict example — illustrative claims, not real dataset results")
demo_claims = []
for record, value, reliability in [
    ("a", "10 King St", 0.8),
    ("b", "10 KING ST", 0.8),
    ("c", "12 King St", 0.4),
]:
    demo_claims.append({
        "claim_id": record, "entity_id": "example_entity", "field_name": "address.full",
        "raw_value": value, "normalized_value": normalize_value("address.full", value),
        "source_id": record, "source_record_id": record,
        "source_reliability": reliability, "link_confidence": 1.0, "field_confidence": 1.0,
        "claim_score": claim_score(reliability, 1.0, 1.0, "example " + record),
        "conflict_id": None,
    })
demo_outputs = reconcile_fields(
    pd.DataFrame({"entity_id": ["example_entity"]}),
    pd.DataFrame(demo_claims, columns=CLAIM_COLUMNS),
)
print("Profile address: null | Address confidence C: null")
display(demo_outputs["entity_conflict_values"][[
    "alternative_id", "value", "support_score", "relative_score"
]])
print("Future decisions for this conflict:")
display(demo_outputs["entity_field_conflicts"][["field_name", *DECISION_COLUMNS]])


PART 4 — Worked conflict example — illustrative claims, not real dataset results
Profile address: null | Address confidence C: null


,alternative_id,value,support_score,relative_score
0,k1,10 KING ST,1.6,0.8
1,k2,12 KING ST,0.4,0.2


Future decisions for this conflict:


,field_name,ai_decision,data_analyser_decision,final_verdict
0,address.full,None,None,None


## Inspect conflicts or a complete entity

The next cell shows up to three real conflicts, with alternatives and source scores. Every question about a future verdict will need this context. Decisions are not requested now.

For any profile, run `inspect_entity("ENTITY_ID")` in the last cell. This prints field values, C, conflict alternatives and the supporting source records without dumping entire tables.

In [9]:
def show_conflict(conflict_id):
    conflict = entity_field_conflicts[entity_field_conflicts["conflict_id"] == conflict_id].iloc[0]
    print(f"\nEntity {conflict['entity_id']} — {conflict['field_name']}")
    print("Profile value: null. Conflict alternatives:")
    alternatives = entity_conflict_values[entity_conflict_values["conflict_id"] == conflict_id]
    for item in alternatives.itertuples(index=False):
        share = "null (zero total support)" if pd.isna(item.relative_score) else f"{item.relative_score:.4f}"
        print(f"  {item.alternative_id}: {item.value} | support={item.support_score:.4f} | relative score={share}")
        supporting = entity_field_claims[
            (entity_field_claims["conflict_id"] == conflict_id)
            & (entity_field_claims["normalized_value"] == item.value)
        ]
        for claim in supporting.itertuples(index=False):
            source = claim.gold_table.removeprefix("gold_").replace("_", " ")
            print(f"    Source: {source}; record: {claim.source_record_id}; observed: {claim.observed_at}")
            print(f"    Raw value: {claim.raw_value}; R={claim.source_reliability:.3f}, L={claim.link_confidence:.3f}, F={claim.field_confidence:.3f}, C={claim.claim_score:.4f}")
    print("AI decision: null | Data analyser decision: null | Final verdict: null")


def inspect_entity(entity_id):
    stage(f"Inspect entity {entity_id}")
    selected = firmable_entities[firmable_entities["entity_id"] == str(entity_id)]
    if selected.empty:
        print("No Part 4 profile for this ID. Check entities_pool or Part 3 unresolved records.")
        return
    profile = selected.iloc[0]
    for field, column in PROFILE_FIELDS.items():
        value, confidence = profile[column], profile[column + "_confidence"]
        if pd.notna(value):
            print(f"{field}: {value} | C={confidence:.4f}")
    missing = json.loads(profile["missing_fields"])
    print("Null profile fields: " + (", ".join(missing) if missing else "none"))
    conflicts = entity_field_conflicts[entity_field_conflicts["entity_id"] == str(entity_id)]
    print(f"Conflicting fields: {len(conflicts)}")
    for conflict_id in conflicts["conflict_id"]:
        show_conflict(conflict_id)


if PART4_READY:
    stage("Conflict preview — alternatives, relative scores and source evidence")
    if entity_field_conflicts.empty:
        print("No field conflicts in the current accepted links.")
    else:
        for conflict_id in entity_field_conflicts.head(3)["conflict_id"]:
            show_conflict(conflict_id)


PART 4 — Conflict preview — alternatives, relative scores and source evidence
No field conflicts in the current accepted links.


In [10]:
if PART4_READY:
    # Replace this ID to inspect another entity, then run this cell.
    ENTITY_TO_INSPECT = str(firmable_entities.iloc[0]["entity_id"]) if len(firmable_entities) else None
    if ENTITY_TO_INSPECT is not None:
        inspect_entity(ENTITY_TO_INSPECT)


PART 4 — Inspect entity 11000000948
entity.legal_name: QBE INSURANCE (INTERNATIONAL) PTY LIMITED | C=1.0000
entity.abn: 11000000948 | C=1.0000
entity.acn: 000000948 | C=1.0000
entity.status: active | C=1.0000
entity.date_registered: 1886-10-01 | C=1.0000
address.country: AU | C=1.0000
Null profile fields: entity.trading_name, entity.nzbn, entity.entity_type, entity.website, entity.industry_code, address.full, address.locality, address.state, address.postcode
Conflicting fields: 0


## Outputs and rerunning

All five tables are saved in `firmable.duckdb` and in `outputs/part4/` as Parquet and CSV; `metrics.json` records the run counts.

| Output | Purpose |
|---|---|
| `firmable_entities` | One profile per entity; uncontested values and C, null conflicting fields |
| `entity_field_scores` | Uncontested field support totals and claim/source counts |
| `entity_field_claims` | Every original linked claim, R/L/F/C, source, timestamps, licence and derivation |
| `entity_field_conflicts` | One conflicting entity/field, three future decision columns |
| `entity_conflict_values` | Each competing value, summed support and relative score |

Run Part 3 again after changing Part 2 gold data or accepted links, then **Run All** here. The same observations count once on every run. This notebook does not generate new entity links or alter the identity registry.

Completeness and conflict counts are measured from the current extracted sample. Human verdicts and measured field accuracy remain future work, as requested; scores do not establish accuracy.